# SED-39 experiments (9 Nov): window length, and rotation/flip augmentation

**Part 1: does a longer window help the size classes?** (about 1 h on a T4) `scripts/window_length_check.py` trains the same small model at 512, 1024, 2048 and 4096 samples on the 21 classes one 160 us window can't settle (QAM, APSK, PSK, ASK sizes and OFDM/LTE/NR), made by our generators with the full impairment chain and a small residual frequency offset. Results: `FOI26/results/2026-11-09_window_length/`.

**Part 2: rotation/flip augmentation on the real model** (about 2 h) The physics model's exact settings (seed 4, expert + spectrum + physics, 15 epochs, fixed data `f993d8ebc417`) plus `--augment`. Then both models are scored on the same fixed test set. Results: `FOI26/results/2026-11-09_physics_augment_seed4/`.

**Colab disconnects:** run all cells again from the top. Part 1 skips lengths already saved; Part 2 resumes from the last saved epoch.

## 1. Settings

In [ ]:
BRANCH = 'eileenyeoh06/sed-39-window-length'
FOI26 = '/content/drive/MyDrive/sedic/FOI26'
WL_OUT = f'{FOI26}/results/2026-11-09_window_length'
DATA_ON_DRIVE = f'{FOI26}/combined_data_labels_v2_fskpskfix'      # f993d8ebc417
RUN_NAME = '2026-11-09_physics_augment'
SEED = 4
EPOCHS = 15
XF = '--expert-features --full-spectrum --physics-features --augment'
CKPT_DIR = f'{FOI26}/results/{RUN_NAME}_seed{SEED}'
BASE_MODEL = f'{FOI26}/results/2026-11-07_physics_seed4/best.pt'   # physics_best.pt, no augmentation

## 2. Mount Drive, get the code, check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf sedicAI_NEXA
!git clone -q -b {BRANCH} https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!git log -1 --format='commit %H  %s'
!pip install -q pyyaml
import torch, os, json
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

# Part 1: window length

## 3. Train at 512 / 1024 / 2048 / 4096 samples (about 1 h)

Each length is saved as soon as it is done (`len_<L>.json`), so a disconnect only loses the one in progress.

In [ ]:
!python scripts/window_length_check.py --out "{WL_OUT}" --workers 2 --per-bin 300 --epochs 10

# Part 2: rotation/flip augmentation

## 4. Copy the fixed data to the Colab disk (a few minutes)

In [ ]:
import numpy as np
if not os.path.exists('/content/combined/combined.json'):
    !mkdir -p /content/combined
    !cp -r "{DATA_ON_DRIVE}/train" "{DATA_ON_DRIVE}/val" "{DATA_ON_DRIVE}/test" /content/combined/
    !cp "{DATA_ON_DRIVE}/combined.json" /content/combined/
print('data version:', json.load(open('/content/combined/combined.json'))['combined_version'], '(expect f993d8ebc417)')
for split, n in (('train', 960_000), ('val', 120_000), ('test', 120_000)):
    assert np.load(f'/content/combined/{split}/X.npy', mmap_mode='r').shape == (n, 2, 512), split

## 5. Train with augmentation (about 2 h)

In [ ]:
!python scripts/train_week1.py --seed {SEED} --data /content/combined --epochs {EPOCHS} --batch-size 256 {XF} --ckpt-dir "{CKPT_DIR}" --eval-test

## 6. Calibration and vote length on val/test (about 3 min)

In [ ]:
!python scripts/calibrate_week1.py --ckpt "{CKPT_DIR}/best.pt" --data /content/combined --cache /content/logits_aug

## 7. Fair comparison on the same fixed test set: physics_best.pt vs + augmentation

Overall, at >= 10 dB, and per family. Writes `compare.json` next to the new model.

In [ ]:
import sys
sys.path.insert(0, '.')
from src.config import DATASET_CLASSES as C, CLASS_GROUPS
from src.models.amc_cnn import AMC_CNN
dev = torch.device('cuda')
FAM = {c: g for g, cs in CLASS_GROUPS.items() for c in cs}; FAM['NOISE_FLOOR'] = 'NOISE'

def load(path):
    ck = torch.load(path, map_location=dev, weights_only=False); m = ck['meta']
    net = AMC_CNN(num_classes=60, input_len=m['input_len'], stft_n_fft=m['stft_n_fft'], **m.get('model_flags', {})).to(dev)
    net.load_state_dict(ck['state_dict'])
    return net.eval()

@torch.no_grad()
def top_of(net, X):
    return np.concatenate([net(torch.from_numpy(np.asarray(X[s:s + 2048], dtype=np.float32)).to(dev)).argmax(1).cpu().numpy()
                           for s in range(0, len(X), 2048)])

X = np.load('/content/combined/test/X.npy', mmap_mode='r'); y = np.load('/content/combined/test/y.npy').astype(bool)
snr = np.load('/content/combined/test/snr_labels.npy')
tf = np.array([FAM[C[i]] for i in y.argmax(1)]); single = y.sum(1) == 1
fams = sorted(set(FAM.values()))
res = {}
for name, path in (('physics_best', BASE_MODEL), ('+ augment', f'{CKPT_DIR}/best.pt')):
    hit = y[np.arange(len(y)), top_of(load(path), X)]
    res[name] = {'top1': float(hit.mean()), 'top1_high': float(hit[snr >= 10].mean()),
                 'family': {g: float(hit[single & (tf == g)].mean()) for g in fams},
                 'per_class': {c: float(hit[single & (y[:, i])].mean()) for i, c in enumerate(C) if (single & y[:, i]).any()}}
print(f"{'':14s}{'physics_best':>14s}{'+ augment':>12s}{'change':>9s}")
for key in ('top1', 'top1_high'):
    a, b = res['physics_best'][key], res['+ augment'][key]
    print(f'{key:14s}{a:14.3f}{b:12.3f}{b - a:+9.3f}')
for g in fams:
    a, b = res['physics_best']['family'][g], res['+ augment']['family'][g]
    print(f'{g:14s}{a:14.3f}{b:12.3f}{b - a:+9.3f}')
json.dump(res, open(f'{CKPT_DIR}/compare.json', 'w'), indent=2)
print('wrote', f'{CKPT_DIR}/compare.json')

## 8. Make sure everything reached Drive

In [ ]:
drive.flush_and_unmount()
print('Drive upload finished')